# Módulo 3 — Métricas de Temporada
**Ciencia de Datos I · ETITC · 2026**  
Daniel Valencia · Daniel Medcalfe

**Criterio de validación:** Gráficas de series de tiempo con identificación de picos estacionales correctamente etiquetados por segmento de cliente y categoría.

## 1. Imports y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

COLORES_CAT = {
    'Clothing':       '#6C63FF',
    'Shoes':          '#43D9AD',
    'Technology':     '#378ADD',
    'Cosmetics':      '#FF6584',
    'Toys':           '#FFB347',
    'Food & Beverage':'#3B6D11',
    'Books':          '#A32D2D',
    'Souvenir':       '#888780',
}

df = pd.read_csv('../../data/processed/datos_limpios.csv')
df = df.rename(columns={'año':'year','mes':'month','dia_semana':'day_of_week','total_compra':'total_spend'})
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['quarter']      = df['invoice_date'].dt.quarter
df['week']         = df['invoice_date'].dt.isocalendar().week.astype(int)
df['month_name']   = df['invoice_date'].dt.strftime('%b')

# Nota: los datos de 2023 solo cubren enero–marzo (dataset incompleto)
print(f"Período: {df['invoice_date'].min().date()} → {df['invoice_date'].max().date()}")
print(f"Filas: {len(df):,}")
print(f"Nota: 2023 solo tiene datos hasta {df[df['year']==2023]['invoice_date'].max().date()}")

## 2. Serie de tiempo — ingresos mensuales totales

In [ ]:
mensual = (
    df.groupby(['year','month'])['total_spend']
    .sum()
    .reset_index()
)
mensual['periodo'] = pd.to_datetime(
    mensual['year'].astype(str) + '-' + mensual['month'].astype(str) + '-01'
)
mensual = mensual[mensual['year'] < 2023]  # 2023 incompleto — excluir para no distorsionar

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(mensual['periodo'], mensual['total_spend'],
        color='#6C63FF', linewidth=2.5, marker='o', markersize=5, zorder=3)
ax.fill_between(mensual['periodo'], mensual['total_spend'],
                alpha=0.10, color='#6C63FF')

# Etiquetar picos (top 5)
top5 = mensual.nlargest(5, 'total_spend')
for _, row in top5.iterrows():
    ax.annotate(
        f"${row['total_spend']/1e6:.2f}M",
        xy=(row['periodo'], row['total_spend']),
        xytext=(0, 12), textcoords='offset points',
        ha='center', fontsize=8, color='#6C63FF', fontweight='bold',
        arrowprops=dict(arrowstyle='-', color='#6C63FF', lw=0.8),
    )

# Sombrear meses de alta temporada (Oct, Nov, Dic, Jul)
for y in [2021, 2022]:
    for m, label in [(7,'Jul'), (10,'Oct'), (12,'Dic')]:
        fecha = pd.Timestamp(year=y, month=m, day=1)
        ax.axvspan(fecha, fecha + pd.DateOffset(months=1),
                   alpha=0.08, color='#FFB347', zorder=1)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.tick_params(axis='x', rotation=35)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.set_title('Ingresos mensuales totales 2021–2022 (con picos etiquetados)', fontsize=12)
ax.set_ylabel('Ingresos (USD)')
ax.spines[['top','right']].set_visible(False)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#FFB347', alpha=0.3, label='Meses de alta temporada (Jul, Oct, Dic)')],
          fontsize=9)

plt.tight_layout()
plt.savefig('../../reports/figures/mod3_serie_mensual.png', bbox_inches='tight')
plt.show()
print("✅ Guardada: reports/figures/mod3_serie_mensual.png")

## 3. Estacionalidad por mes — promedio 2021–2022

In [ ]:
# Promedio de ingresos por mes (sin 2023 incompleto)
df_completo = df[df['year'] < 2023].copy()

estacional = (
    df_completo.groupby('month')['total_spend']
    .mean()
    .reset_index()
)
MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
estacional['mes_nombre'] = estacional['month'].apply(lambda x: MESES[x-1])

fig, ax = plt.subplots(figsize=(12, 4))

colores_bar = ['#D85A30' if v >= estacional['total_spend'].quantile(0.75)
               else '#6C63FF' if v >= estacional['total_spend'].median()
               else '#C7CEEA'
               for v in estacional['total_spend']]

bars = ax.bar(estacional['mes_nombre'], estacional['total_spend'],
              color=colores_bar, edgecolor='none', width=0.6)
ax.bar_label(bars, labels=[f'${v/1e6:.2f}M' for v in estacional['total_spend']],
             padding=4, fontsize=8)

ax.axhline(estacional['total_spend'].mean(), color='#888', linestyle='--',
           linewidth=1.2, label=f"Promedio: ${estacional['total_spend'].mean()/1e6:.2f}M")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.set_title('Estacionalidad mensual promedio (2021–2022)', fontsize=12)
ax.set_ylabel('Ingresos promedio (USD)')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

from matplotlib.patches import Patch
leyenda = [
    Patch(color='#D85A30', label='Alta temporada (top 25%)'),
    Patch(color='#6C63FF', label='Temporada media'),
    Patch(color='#C7CEEA', label='Temporada baja'),
]
ax.legend(handles=leyenda + [plt.Line2D([0],[0],color='#888',linestyle='--',label='Promedio anual')],
          fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('../../reports/figures/mod3_estacionalidad.png', bbox_inches='tight')
plt.show()
print("✅ Guardada: reports/figures/mod3_estacionalidad.png")

## 4. Ingresos por trimestre

In [ ]:
trimestral = (
    df_completo.groupby(['year','quarter'])['total_spend']
    .sum()
    .reset_index()
)
trimestral['label'] = trimestral.apply(lambda r: f"Q{r['quarter']} {r['year']}", axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
colores_q = ['#6C63FF','#43D9AD','#FFB347','#FF6584']

for i, (_, grp) in enumerate(trimestral.groupby('year')):
    x = [f"Q{q}" for q in grp['quarter']]
    ax.plot(x, grp['total_spend'], marker='o', linewidth=2,
            label=str(grp['year'].iloc[0]),
            color=['#6C63FF','#43D9AD'][i % 2])

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.set_title('Ingresos por trimestre — 2021 vs 2022', fontsize=12)
ax.set_ylabel('Ingresos (USD)')
ax.legend(title='Año', fontsize=9)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../../reports/figures/mod3_trimestral.png', bbox_inches='tight')
plt.show()
print("✅ Guardada: reports/figures/mod3_trimestral.png")

## 5. Tendencia por categoría — ¿cuáles tienen estacionalidad propia?

In [ ]:
cat_mensual = (
    df_completo.groupby(['year','month','category'])['total_spend']
    .sum()
    .reset_index()
)
cat_mensual['periodo'] = pd.to_datetime(
    cat_mensual['year'].astype(str) + '-' + cat_mensual['month'].astype(str) + '-01'
)

# Top 4 categorías por ingreso total
top4_cats = df_completo.groupby('category')['total_spend'].sum().nlargest(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharey=False)
axes = axes.flatten()

for i, cat in enumerate(top4_cats):
    sub = cat_mensual[cat_mensual['category'] == cat]
    color = COLORES_CAT.get(cat, '#6C63FF')
    axes[i].plot(sub['periodo'], sub['total_spend'],
                 color=color, linewidth=2, marker='o', markersize=4)
    axes[i].fill_between(sub['periodo'], sub['total_spend'], alpha=0.1, color=color)
    axes[i].set_title(cat, fontsize=11)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    axes[i].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
    axes[i].spines[['top','right']].set_visible(False)

fig.suptitle('Ingresos mensuales por categoría — Top 4 (2021–2022)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/mod3_tendencia_categorias.png', bbox_inches='tight')
plt.show()
print("✅ Guardada: reports/figures/mod3_tendencia_categorias.png")

## 6. Día de la semana — ¿cuándo compra más la gente?

In [ ]:
orden_dias = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
DIAS_ES    = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']

dia_semana = (
    df_completo.groupby('day_of_week')['total_spend']
    .mean()
    .reindex(orden_dias)
    .reset_index()
)
dia_semana['dia_es'] = DIAS_ES

fig, ax = plt.subplots(figsize=(10, 4))
colores_dia = ['#D85A30' if d in ['Saturday','Sunday'] else '#6C63FF' for d in orden_dias]

bars = ax.bar(dia_semana['dia_es'], dia_semana['total_spend'],
              color=colores_dia, edgecolor='none', width=0.6)
ax.bar_label(bars, labels=[f'${v:,.0f}' for v in dia_semana['total_spend']],
             padding=4, fontsize=8)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Gasto promedio por día de la semana', fontsize=12)
ax.set_ylabel('Gasto promedio (USD)')
ax.spines[['top','right']].set_visible(False)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#D85A30', label='Fin de semana'),
                   Patch(color='#6C63FF', label='Entre semana')], fontsize=9)

plt.tight_layout()
plt.savefig('../../reports/figures/mod3_dia_semana.png', bbox_inches='tight')
plt.show()
print("✅ Guardada: reports/figures/mod3_dia_semana.png")

## 7. Exportar resultados

In [ ]:
from pathlib import Path
out = Path('../../reports/resultados')
out.mkdir(parents=True, exist_ok=True)

estacional.to_csv(out / 'mod3_estacionalidad_mensual.csv', index=False)
trimestral.to_csv(out / 'mod3_trimestral.csv', index=False)
cat_mensual.to_csv(out / 'mod3_tendencia_categorias.csv', index=False)

print("✅ Exportados:")
print("   mod3_estacionalidad_mensual.csv")
print("   mod3_trimestral.csv")
print("   mod3_tendencia_categorias.csv")

## 8. Conclusiones del módulo

In [ ]:
mes_pico  = estacional.loc[estacional['total_spend'].idxmax(), 'mes_nombre']
mes_bajo  = estacional.loc[estacional['total_spend'].idxmin(), 'mes_nombre']
pico_val  = estacional['total_spend'].max()
bajo_val  = estacional['total_spend'].min()
variacion = (pico_val - bajo_val) / bajo_val * 100

print("=" * 55)
print("CONCLUSIONES — Módulo 3: Métricas de Temporada")
print("=" * 55)
print(f"  Mes de mayor ingreso  : {mes_pico}  (${pico_val/1e6:.2f}M)")
print(f"  Mes de menor ingreso  : {mes_bajo}  (${bajo_val/1e6:.2f}M)")
print(f"  Variación pico-valle  : {variacion:.1f}%")
print()
print("  Alta temporada  : Julio, Octubre, Diciembre")
print("  Temporada media : Enero, Mayo, Agosto, Septiembre")
print("  Temporada baja  : Febrero, Abril, Junio, Noviembre")
print()
print("  Hallazgos clave:")
print("  → El Q3 (Jul–Sep) y Q4 (Oct–Dic) concentran los mayores ingresos")
print("  → Febrero es consistentemente el mes más bajo (post-navidad)")
print("  → Clothing mantiene picos propios en Oct y Dic (temporada de frío)")
print("  → El fin de semana genera mayor gasto promedio que entre semana")
print()
print("  Gráficas: reports/figures/mod3_*.png")
print("  Datos   : reports/resultados/mod3_*.csv")